In [1]:
import torch
import random
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

!curl -O https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
words = open('names.txt', 'r').read().splitlines()

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  222k  100  222k    0     0   974k      0 --:--:-- --:--:-- --:--:-- 1003k


In [2]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
block_size = 3

In [3]:
def build_dataset(words):
  X, Y = [], []
  for w in words:
     context = [0] * block_size
     for ch in w + '.':
       ix= stoi[ch]
       X.append(context)
       Y.append(ix)
       context = context[1:] + [ix]
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


In [4]:
def cmp(s,dt,t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [5]:
n_embd = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
  p.requires_grad = True

In [6]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [7]:
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)
hprebn = embcat @ W1 + b1
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
h = torch.tanh(hpreact)
logits = h @ W2 + b2
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()
#forward pass
#logprobs matrisindeki elemanların loss üzerindeki etkisi ne?
#Doğru sınıflar için türev -1/n, yanlışlar için 0.

for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3355, grad_fn=<NegBackward0>)

In [8]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
#logaritmada türev 1/içerdeki değer
dprobs = (1.0 / probs) * dlogprobs
#counts.counts_sum_inv şeklinde bundan sumla toplıycaz.
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
#Bu şey mantığı; counts A olsa counts_sum_inv B olsa A ya göre türev B bundan çarpıyoruz
dcounts = counts_sum_inv * dprobs
#x üzeri -1 in türevi nedir diye sorduğumuzda aslında cevabı buluyoruz.
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
#Aynı değişken iki farklı yolda kullanıldağı için türevler toplanır.
dcounts += torch.ones_like(counts) * dcounts_sum
#Üslü ifadenin türevi yine kendisidir.
dnorm_logits = counts * dcounts
#Bu sefer de logits'a A diyelim logit_maxes de B olsun. Buna göre A'ya göre türev ilk başta aşağıdaki gibi gelir;
dlogits = dnorm_logits.clone()
#Bu da B ye görev türev sütun boyunca toplanır ondan sum.
dlogits_maxes = (-dnorm_logits).sum(1, keepdim=True)
# logit_maxes, logits.max(1) ile seçilen maksimum elemanlara 1 dağıtır.
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogits_maxes
#h: (n, n_hidden),(32, 64)
#W2: (n_hidden, vocab_size),(64, 27)
#b2: (vocab_size,),(27,)
#logits: (n, vocab_size),(32, 27)
#dh boyutu(32,64) olmalı
#dlogits: (32, 27)
#W2.T: (27, 64)
#(32, 27) @ (27, 64) = (32, 64)
dh = dlogits @ W2.T
#dW2 boyutu (64, 27) olmalı.
#h.T: (64, 32)
#dlogits: (32, 27)
dW2 = h.T @ dlogits
#Broadcasting ile genişletilen boyutlar toplanır bu yüzden; çünkü b2 tek vektör(27,)
db2 = dlogits.sum(0)
#tanh in türevi ve zincir kuralından dh ile çarpım
dhpreact = (1.0 - h**2) * dh
#bnraw ile çarpıldığından türevi bnraw satır boyunca toplanır.
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
#bngainle çarpıldığı için
dbnraw = bngain * dhpreact
#Toplama işleminin türevi 1 sonrasında satır boyunca toplanır.
dbnbias = dhpreact.sum(0, keepdim=True)
#Yine A.B mantığından dolayı türevi böyle
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
#Burda da klasik türevden üssü başa getirip 1 azaltıyoruz ve zincir kuralından dolayı çarpım yapıyoruz.
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
#1/(n-1) ile çarpıyoruz
dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
#Birden fazla yerde kullandığımız için topluyoruz.
dbndiff += (2*bndiff) * dbndiff2
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)
#(32, 64)@(64, 30) = (32, 30)
dembcat = dhprebn @ W1.T
#(30, 32)@(32, 64) = (30, 64)
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogits_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('emb', demb, emb)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10
bngain          | exact: False | approximate: True  | maxdiff: 3.725290298461914e-09
bnbias          | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09
bnraw   

In [9]:
dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

In [10]:
dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))

In [11]:
max_steps = 1000
n = batch_size
lr = 0.1
lossi = []
for i in range(max_steps):

    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    hprebn = embcat @ W1 + b1
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # dlogits (Softmax + CE)
    dlogits = F.softmax(logits, dim=1)
    dlogits[range(n), Yb] -= 1.0
    dlogits /= n

    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)

    # Tanh backprop
    dhpreact = (1.0 - h**2) * dh

    # BatchNorm parametreleri
    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)

    # BatchNorm girdi türevi
    dhprebn = bngain * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0) - n/(n-1) * bnraw * (dhpreact * bnraw).sum(0))


    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)

    demb = dembcat.view(emb.shape)
    dC = torch.zeros_like(C)
    for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            ix = Xb[k, j]
            dC[ix] += demb[k, j]


    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    for p, grad in zip(parameters, grads):
        p.data -= lr * grad
    lossi.append(loss.item())

    if i % 100 == 0:
       avg = sum(lossi[-100:]) / len(lossi[-100:])
       print(f'Step {i:4d}/{max_steps:4d} | Loss: {avg:.4f}')

Step    0/1000 | Loss: 3.4996
Step  100/1000 | Loss: 3.0003
Step  200/1000 | Loss: 2.6999
Step  300/1000 | Loss: 2.6254
Step  400/1000 | Loss: 2.5532
Step  500/1000 | Loss: 2.5768
Step  600/1000 | Loss: 2.5315
Step  700/1000 | Loss: 2.4935
Step  800/1000 | Loss: 2.5019
Step  900/1000 | Loss: 2.4479
